In [14]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
import os

# 创建输出目录
output_dir = "regression_results"
os.makedirs(output_dir, exist_ok=True)

# 读取数据
panel = pd.read_csv("C:/Users/31314/Desktop/事件研究5/2最终面板数据_with_AR_CAR.csv")
raw = pd.read_csv("C:/Users/31314/Desktop/为上市公司匹配ticker并清洗脏数据3/5手动匹配错误公司并删除后.csv", encoding="gbk")

# 数据处理
panel["event_dt"] = pd.to_datetime(panel["event_date"], errors="coerce")
raw["event_dt"] = pd.to_datetime(raw["announcement_date_verified"], errors="coerce")

event_panel = (
    panel.sort_values(["event_id", "t"])
    .groupby("event_id")
    .first()
    .reset_index()[["event_id", "ticker", "company", "event_dt", 
                    "CAR_CAPM_short_term", "CAR_FF4_short_term"]]
)

raw_agg = (
    raw.sort_values(["ticker", "event_dt"])
    .groupby(["ticker", "event_dt"], as_index=False)
    .agg({
        "percentage_laid_off": "max",
        "funds_raised": "max",
        "ai_mentioned": "max",
        "company": "first"
    })
)

panel["ret_num"] = pd.to_numeric(panel["ret"], errors="coerce")
prior = (
    panel[(panel["t"] >= -126) & (panel["t"] <= -1)]
    .groupby("event_id")
    .agg(
        prior_6m_return=("ret_num", lambda x: np.prod(1 + x.dropna()) - 1 if x.notna().sum() > 0 else np.nan),
        n_pre=("ret_num", "count")
    )
    .reset_index()
)

# 合并数据
reg = event_panel.merge(raw_agg, on=["ticker", "event_dt"], how="left").merge(
    prior[["event_id", "prior_6m_return", "n_pre"]], on="event_id", how="left"
)

# 数据清洗和转换
reg["layoff_pct"] = pd.to_numeric(reg["percentage_laid_off"], errors="coerce")
reg["funds_raised_num"] = pd.to_numeric(reg["funds_raised"], errors="coerce")
reg["log_funds_raised"] = np.log(reg["funds_raised_num"].where(reg["funds_raised_num"] > 0, 1))
reg["year"] = reg["event_dt"].dt.year.astype(int)

# 删除缺失值
complete = reg.dropna(subset=[
    "CAR_CAPM_short_term", "CAR_FF4_short_term", "layoff_pct",
    "ai_mentioned", "log_funds_raised", "prior_6m_return", "year"
]).copy()

print(f"最终回归样本大小: {len(complete)} 个观测值")
print(f"描述性统计:")
print(complete[['CAR_CAPM_short_term', 'CAR_FF4_short_term', 'layoff_pct', 
                'ai_mentioned', 'log_funds_raised', 'prior_6m_return']].describe())

# 运行回归
m_capm = smf.ols(
    "CAR_CAPM_short_term ~ layoff_pct + ai_mentioned + log_funds_raised + prior_6m_return + C(year)",
    data=complete
).fit(cov_type="HC3")

m_ff4 = smf.ols(
    "CAR_FF4_short_term ~ layoff_pct + ai_mentioned + log_funds_raised + prior_6m_return + C(year)",
    data=complete
).fit(cov_type="HC3")


# 2. 将摘要报告保存到文本文件
with open(f"{output_dir}/capm_summary.txt", "w", encoding="utf-8") as f:
    f.write("CAPM模型回归结果摘要\n")
    f.write("="*80 + "\n")
    f.write(str(m_capm.summary()))
    f.write(f"\n\n样本大小: {len(complete)}")
    f.write(f"\nR²: {m_capm.rsquared:.4f}")
    f.write(f"\n调整R²: {m_capm.rsquared_adj:.4f}")
    f.write(f"\nF统计量: {m_capm.fvalue:.4f}")
    f.write(f"\nF统计量p值: {m_capm.f_pvalue:.4f}")

with open(f"{output_dir}/ff4_summary.txt", "w", encoding="utf-8") as f:
    f.write("FF4模型回归结果摘要\n")
    f.write("="*80 + "\n")
    f.write(str(m_ff4.summary()))
    f.write(f"\n\n样本大小: {len(complete)}")
    f.write(f"\nR²: {m_ff4.rsquared:.4f}")
    f.write(f"\n调整R²: {m_ff4.rsquared_adj:.4f}")
    f.write(f"\nF统计量: {m_ff4.fvalue:.4f}")
    f.write(f"\nF统计量p值: {m_ff4.f_pvalue:.4f}")

# 创建并排表格
results_table = summary_col(
    [m_capm, m_ff4],
    model_names=['CAPM', 'FF4'],
    stars=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs):d}",
        'R2': lambda x: f"{x.rsquared:.4f}",
        'Adj.R2': lambda x: f"{x.rsquared_adj:.4f}",
        'F-stat': lambda x: f"{x.fvalue:.2f}"
    },
    regressor_order=['Intercept', 'layoff_pct', 'ai_mentioned', 
                     'log_funds_raised', 'prior_6m_return']
)

print(results_table)

# 保存并排表格为文本
with open(f"{output_dir}/side_by_side_results.txt", "w", encoding="utf-8") as f:
    f.write("CAPM和FF4模型并排回归结果\n")
    f.write("="*100 + "\n")
    f.write(str(results_table))

# 4. 提取系数、标准误、t值、p值到DataFrame
def extract_coefficients(model, model_name):
    """从模型结果中提取系数信息"""
    coef_df = pd.DataFrame({
        'coefficient': model.params,
        'std_err': model.bse,
        't_value': model.tvalues,
        'p_value': model.pvalues,
        'conf_lower': model.conf_int()[0],
        'conf_upper': model.conf_int()[1]
    })
    
    # 添加显著性星号
    coef_df['significance'] = ''
    coef_df.loc[coef_df['p_value'] < 0.01, 'significance'] = '***'
    coef_df.loc[(coef_df['p_value'] >= 0.01) & (coef_df['p_value'] < 0.05), 'significance'] = '**'
    coef_df.loc[(coef_df['p_value'] >= 0.05) & (coef_df['p_value'] < 0.10), 'significance'] = '*'
    
    coef_df['model'] = model_name
    coef_df = coef_df.reset_index().rename(columns={'index': 'variable'})
    
    return coef_df

# 提取两个模型的系数
capm_coef = extract_coefficients(m_capm, 'CAPM')
ff4_coef = extract_coefficients(m_ff4, 'FF4')

# 保存系数表格为CSV
capm_coef.to_csv(f"{output_dir}/capm_coefficients.csv", index=False, encoding='utf-8-sig')
ff4_coef.to_csv(f"{output_dir}/ff4_coefficients.csv", index=False, encoding='utf-8-sig')

# 合并系数
all_coef = pd.concat([capm_coef, ff4_coef], ignore_index=True)
all_coef.to_csv(f"{output_dir}/all_coefficients.csv", index=False, encoding='utf-8-sig')

# 5. 创建模型统计量汇总
model_stats = pd.DataFrame({
    'Model': ['CAPM', 'FF4'],
    'N': [int(m_capm.nobs), int(m_ff4.nobs)],
    'R2': [m_capm.rsquared, m_ff4.rsquared],
    'Adj_R2': [m_capm.rsquared_adj, m_ff4.rsquared_adj],
    'F_Statistic': [m_capm.fvalue, m_ff4.fvalue],
    'F_p_value': [m_capm.f_pvalue, m_ff4.f_pvalue],
    'AIC': [m_capm.aic, m_ff4.aic],
    'BIC': [m_capm.bic, m_ff4.bic],
    'Log_Likelihood': [m_capm.llf, m_ff4.llf]
})

model_stats.to_csv(f"{output_dir}/model_statistics.csv", index=False, encoding='utf-8-sig')

# 7. 保存回归数据
complete.to_csv(f"{output_dir}/regression_sample_data.csv", index=False, encoding='utf-8-sig')

# 10. 创建详细的Excel工作簿
with pd.ExcelWriter(f"{output_dir}/regression_results.xlsx", engine='openpyxl') as writer:
    # 保存模型统计
    model_stats.to_excel(writer, sheet_name='模型统计', index=False)
    
    # 保存CAPM系数
    capm_coef.to_excel(writer, sheet_name='CAPM系数', index=False)
    
    # 保存FF4系数
    ff4_coef.to_excel(writer, sheet_name='FF4系数', index=False)
    
    # 保存系数比较
    if 'coef_comparison' in locals():
        coef_comparison.to_excel(writer, sheet_name='系数比较', index=False)
    
    # 保存回归样本数据
    complete.to_excel(writer, sheet_name='回归样本', index=False)
    
    # 保存描述性统计
    desc_stats = complete[['CAR_CAPM_short_term', 'CAR_FF4_short_term', 'layoff_pct', 
                          'ai_mentioned', 'log_funds_raised', 'prior_6m_return']].describe()
    desc_stats.to_excel(writer, sheet_name='描述性统计')



最终回归样本大小: 249 个观测值
描述性统计:
       CAR_CAPM_short_term  CAR_FF4_short_term  layoff_pct  ai_mentioned  \
count           249.000000          249.000000  249.000000    249.000000   
mean              0.103832            0.097966    0.128273      0.582329   
std               0.238362            0.228041    0.142137      0.494169   
min              -0.976623           -1.004382    0.000000      0.000000   
25%               0.047550            0.040800    0.050000      0.000000   
50%               0.102180            0.099083    0.100000      1.000000   
75%               0.176861            0.180628    0.150000      1.000000   
max               0.794171            0.752653    1.000000      1.000000   

       log_funds_raised  prior_6m_return  
count        249.000000       249.000000  
mean           5.369627        -0.282980  
std            2.465171         0.351462  
min            0.000000        -2.387315  
25%            4.753590        -0.516266  
50%            6.061457        